In [1]:
from llms.genserv.model_generator_vllm import AsyncVLLMGenerationModel

model_name = "microsoft/phi-4"
model = AsyncVLLMGenerationModel(model_name=model_name, enable_prefix_caching=True, max_context_length=1500)

The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


WARNING 01-02 08:29:40 [model.py:2005] Casting torch.bfloat16 to torch.float16.
WARNING 01-02 08:29:40 [arg_utils.py:1869] This model does not officially support disabling chunked prefill. Disabling this manually may cause the engine to crash or produce incorrect outputs.


2026-01-02 08:29:40,906	INFO util.py:154 -- Outdated packages:
  ipywidgets==7.8.1 found, needs ipywidgets>=8
Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
Loading safetensors checkpoint shards:   0% Completed | 0/6 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  17% Completed | 1/6 [00:01<00:07,  1.40s/it]
Loading safetensors checkpoint shards:  33% Completed | 2/6 [00:02<00:06,  1.50s/it]
Loading safetensors checkpoint shards:  50% Completed | 3/6 [00:04<00:04,  1.52s/it]
Loading safetensors checkpoint shards:  67% Completed | 4/6 [00:06<00:03,  1.57s/it]
Loading safetensors checkpoint shards:  83% Completed | 5/6 [00:07<00:01,  1.58s/it]
Loading safetensors checkpoint shards: 100% Completed | 6/6 [00:09<00:00,  1.59s/it]
Loading safetensors checkpoint shards: 100% Completed | 6/6 [00:09<00:00,  1.56s/it]
(EngineCore_DP0 pid=3675677) 
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 35/35 [00:03<00:00, 10

Async vLLM model loaded: microsoft/phi-4 (prefix caching: enabled, max context: 1500)


In [2]:
from tasks import get_task
import json, time, numpy as np

task_id = "sharded-livecodebench/2857"
dataset_fn = "data/sharded_instructions_600.json"
with open(dataset_fn, "r") as f:
    data = json.load(f)

data = [d for d in data if d["task"] == "code"]
sample = [d for d in data if d["task_id"] == task_id][0]

task = get_task(sample["task"])

system_message = task.generate_system_prompt(sample)
input_prompt = task.populate_fully_specific_prompt(sample)

conversation = [{"role": "system", "content": system_message}, {"role": "user", "content": input_prompt}]

max_tokens = 1000
temperature = 0.7
top_k = 20

num_responses_list = [10, 100, 1000]
# num_responses_list = [5000]
num_trials = 5

In [3]:
# Benchmark compressed IID sampling
compressed_results = []

for n in num_responses_list:
    print(f"\n{'='*50}")
    print(f"Compressed IID Sampling: {n} responses ({num_trials} trials)")
    print('='*50)
    
    trial_times = []
    trial_logprobs = []
    
    for trial in range(num_trials):
        start = time.time()
        results = await model.generate_compressed_iid_sampling(conversation, num_responses=n, max_tokens=max_tokens, temperature=temperature, top_k=top_k)
        elapsed = time.time() - start
        
        trial_times.append(elapsed)
        trial_logprobs.append(np.mean([r['logprobs'] for r in results]))
        print(f"  Trial {trial+1}: {elapsed:.2f}s")
    
    avg_time = np.mean(trial_times)
    std_time = np.std(trial_times)
    avg_logprob = np.mean(trial_logprobs)
    time_per_response = avg_time / n
    
    compressed_results.append({'n': n, 'time': avg_time, 'time_std': std_time, 'time_per_response': time_per_response, 'avg_logprob': avg_logprob})
    
    print(f"Avg Time: {avg_time:.2f}s ± {std_time:.2f}s | Per response: {time_per_response*1000:.2f}ms | Avg logprob: {avg_logprob:.3f}")



Compressed IID Sampling: 10 responses (5 trials)
Compressed IID sampling complete in 13.53s: 10 branches, 10 total samples
Efficiency: 1592 naive tokens vs 1347 actual (1.2x savings)
  Trial 1: 13.53s
Compressed IID sampling complete in 12.24s: 10 branches, 10 total samples
Efficiency: 1577 naive tokens vs 1332 actual (1.2x savings)
  Trial 2: 12.24s
Compressed IID sampling complete in 11.76s: 10 branches, 10 total samples
Efficiency: 1594 naive tokens vs 1349 actual (1.2x savings)
  Trial 3: 11.76s
Compressed IID sampling complete in 13.83s: 10 branches, 10 total samples
Efficiency: 1589 naive tokens vs 1344 actual (1.2x savings)
  Trial 4: 13.83s
Compressed IID sampling complete in 11.79s: 10 branches, 10 total samples
Efficiency: 1614 naive tokens vs 1369 actual (1.2x savings)
  Trial 5: 11.79s
Avg Time: 12.63s ± 0.88s | Per response: 1262.93ms | Avg logprob: -24.185

Compressed IID Sampling: 100 responses (5 trials)
Compressed IID sampling complete in 20.22s: 99 branches, 100 tota

In [4]:
# Benchmark true IID sampling
true_iid_results = []

for n in num_responses_list:
    print(f"\n{'='*50}")
    print(f"True IID Sampling: {n} responses ({num_trials} trials)")
    print('='*50)
    
    trial_times = []
    trial_logprobs = []
    
    for trial in range(num_trials):
        start = time.time()
        results = await model.generate_batch_async([conversation], n_responses_per_conv=n, max_tokens=max_tokens, temperature=temperature, top_k=top_k)
        elapsed = time.time() - start
        
        results = results[0]  # unwrap single conversation
        trial_times.append(elapsed)
        trial_logprobs.append(np.mean([r['logprobs'] for r in results]))
        print(f"  Trial {trial+1}: {elapsed:.2f}s")
    
    avg_time = np.mean(trial_times)
    std_time = np.std(trial_times)
    avg_logprob = np.mean(trial_logprobs)
    time_per_response = avg_time / n
    
    true_iid_results.append({'n': n, 'time': avg_time, 'time_std': std_time, 'time_per_response': time_per_response, 'avg_logprob': avg_logprob})
    
    print(f"Avg Time: {avg_time:.2f}s ± {std_time:.2f}s | Per response: {time_per_response*1000:.2f}ms | Avg logprob: {avg_logprob:.3f}")



True IID Sampling: 10 responses (5 trials)
  Trial 1: 9.74s
  Trial 2: 11.12s
  Trial 3: 8.63s
  Trial 4: 8.88s
  Trial 5: 9.33s
Avg Time: 9.54s ± 0.88s | Per response: 953.82ms | Avg logprob: -20.844

True IID Sampling: 100 responses (5 trials)
  Trial 1: 14.73s
  Trial 2: 15.36s
  Trial 3: 14.78s
  Trial 4: 14.89s
  Trial 5: 14.31s
Avg Time: 14.81s ± 0.34s | Per response: 148.15ms | Avg logprob: -22.441

True IID Sampling: 1000 responses (5 trials)
  Trial 1: 110.48s
  Trial 2: 110.34s
  Trial 3: 108.74s
  Trial 4: 109.58s
  Trial 5: 107.84s
Avg Time: 109.40s ± 0.99s | Per response: 109.40ms | Avg logprob: -22.894


In [5]:
# Compare results
print("\n" + "="*100)
print(f"COMPARISON: Compressed IID vs True IID Sampling ({num_trials} trials each)")
print("="*100)
print(f"{'N':>6} | {'Compressed':>18} | {'True IID':>18} | {'Speedup':>8} | {'Comp LP':>10} | {'True LP':>10}")
print("-"*100)

for comp, true in zip(compressed_results, true_iid_results):
    n = comp['n']
    comp_str = f"{comp['time']:.2f}s ± {comp['time_std']:.2f}s"
    true_str = f"{true['time']:.2f}s ± {true['time_std']:.2f}s"
    speedup = true['time'] / comp['time']
    print(f"{n:>6} | {comp_str:>18} | {true_str:>18} | {speedup:>7.2f}x | {comp['avg_logprob']:>10.2f} | {true['avg_logprob']:>10.2f}")



COMPARISON: Compressed IID vs True IID Sampling (5 trials each)
     N |         Compressed |           True IID |  Speedup |    Comp LP |    True LP
----------------------------------------------------------------------------------------------------
    10 |     12.63s ± 0.88s |      9.54s ± 0.88s |    0.76x |     -24.18 |     -20.84
   100 |     19.38s ± 0.55s |     14.81s ± 0.34s |    0.76x |     -22.36 |     -22.44
  1000 |     90.18s ± 1.61s |    109.40s ± 0.99s |    1.21x |     -22.18 |     -22.89
